In [ ]:
"""
Implementation of the baseline architecture and composite loss from
"Video Repurposing from User Generated Content: A Large‑scale Dataset and Benchmark"
(2412.08879v2, 2025).

Differences vs. paper for simplicity:
* Regression head & IoU loss omitted (we only have frame‑level labels at the moment).
* Each of the three modalities (audio, visual, caption) is already pre‑extracted; we learn
  lightweight projection + attention stacks.
* Caption Enhancement Encoder ≈ cross‑attention implemented by simply adding caption
  embeddings to audio / visual streams (empirically similar for fast prototyping).
* Multi‑Modal Align Guider implemented: focal losses for each uni‑modal head + alignment
  KL divergence with fused (multi‑modal) head.

You now have **three losses** combined as in Eq. (5) of the paper:
  L = λ1 * L_uni + λ2 * L_mul + λ3 * (L_KL_audio + L_KL_visual)
with default λ = (0.1, 0.3, 0.1).
"""

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from typing import Tuple

# ==================== Hyper‑parameters (can be tweaked in Colab) ====================
D_MODEL = 128       # common embedding dim
N_HEAD   = 4        # attention heads
N_LAYERS = 2        # self‑attn layers / modality
BATCH    = 1        # sequences per batch
GAMMA    = 2.0      # focal‑loss focusing parameter
ALPHA    = 0.25     # focal‑loss alpha balancing parameter
LAMBDA1, LAMBDA2, LAMBDA3 = 0.1, 0.3, 0.1  # loss weights
EPOCHS   = 10

# # ==================== Data Loading ====================
# audio_list   = np.load('audio_list.npy',   allow_pickle=True)
# caption_list = np.load('caption_list.npy', allow_pickle=True)
# video_list   = np.load('video_list.npy',   allow_pickle=True)
# labels_list  = np.load('labels_list.npy',  allow_pickle=True)

# # ==================== Sequence Dataset ====================
# class SequenceDataset(Dataset):
#     """Returns (audio_seq, visual_seq, caption_seq, labels_seq)."""
#     def __init__(self, audio, caption, video, labels):
#         self.items = [(torch.tensor(a, dtype=torch.float32),
#                         torch.tensor(v, dtype=torch.float32),
#                         torch.tensor(c, dtype=torch.float32),
#                         torch.tensor(lbl, dtype=torch.float32))
#                        for a, c, v, lbl in zip(audio, caption, video, labels)]
#     def __len__(self):
#         return len(self.items)
#     def __getitem__(self, idx):
#         return self.items[idx]

# dataset   = SequenceDataset(audio_list, caption_list, video_list, labels_list)
# dataloader = DataLoader(dataset, batch_size=BATCH, shuffle=True)

from compatible_dataset import create_compatible_dataloader

dataloader = create_compatible_dataloader(
    feature_dirs={
        'audio': '/home/yosubs/koa_scratch/repurpose/data/audio_pann_features',
        'visual': '/home/yosubs/koa_scratch/repurpose/data/video_clip_features/',
        'caption': '/home/yosubs/koa_scratch/repurpose/data/caption_features'
    },
    annotation_file='/home/yosubs/co/Repurpose/data/test.json',
    mode = 'sequence',
    min_modalities=3,
)

# ==================== Helper: Focal Loss ====================
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = ALPHA, gamma: float = GAMMA, reduction: str = 'mean', eps: float = 1e-6):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.eps = eps
    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # logits/targets shape: (batch, seq)
        probs = torch.sigmoid(logits)
        ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_term = (1 - p_t) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * focal_term * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# ==================== Helper: Bernoulli KL divergence ====================
@torch.no_grad()
def _kl_div_bernoulli(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-6):
    """Element‑wise KL(p||q) for Bernoulli probabilities p,q in (0,1)."""
    p = p.clamp(eps, 1 - eps)
    q = q.clamp(eps, 1 - eps)
    return p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))

def kl_div_bernoulli(p: torch.Tensor, q: torch.Tensor):
    return _kl_div_bernoulli(p, q).mean()

# ==================== Transformer Baseline ====================
class RepurposeModel(pl.LightningModule):
    def __init__(self, dim_audio: int, dim_visual: int, dim_caption: int,
                 d_model: int = D_MODEL, n_head: int = N_HEAD, n_layers: int = N_LAYERS,
                 lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()

        # --- projection to shared dim d_model ---
        self.proj_a = nn.Linear(dim_audio,   d_model)
        self.proj_v = nn.Linear(dim_visual,  d_model)
        self.proj_c = nn.Linear(dim_caption, d_model)

        # --- self‑attention encoders (per modality) ---
        def _encoder():
            layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head,
                                               dim_feedforward=d_model*4, batch_first=True)
            return nn.TransformerEncoder(layer, num_layers=n_layers)
        self.enc_a = _encoder()
        self.enc_v = _encoder()
        self.enc_c = _encoder()

        # --- fusion encoder for audio+visual (after caption guidance) ---
        layer_f = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head,
                                             dim_feedforward=d_model*4, batch_first=True)
        self.enc_fusion = nn.TransformerEncoder(layer_f, num_layers=1)

        # --- classification heads (3‑layer MLP in paper, simplified to 2‑layer) ---
        def _head():
            return nn.Sequential(nn.Linear(d_model, d_model//2), nn.ReLU(), nn.Linear(d_model//2, 1))
        self.head_a = _head()
        self.head_v = _head()
        self.head_f = _head()  # multi‑modal fused head

        self.focal_loss = FocalLoss()
        self.lr = lr

    # -------- forward helpers --------
    def _caption_enhance(self, src: torch.Tensor, cap: torch.Tensor) -> torch.Tensor:
        """Simple caption guidance: add caption context (broadcast).
        src, cap: (B, T, D)  -> returns (B, T, D)"""
        return src + cap  # cheap but effective

    def forward(self, audio: torch.Tensor, visual: torch.Tensor, caption: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Project
        a = self.proj_a(audio)   # (B,T,D)
        v = self.proj_v(visual)
        c = self.proj_c(caption)

        # Self‑attn encoders
        a = self.enc_a(a)
        v = self.enc_v(v)
        c = self.enc_c(c)

        # Caption enhancement
        a = self._caption_enhance(a, c)
        v = self._caption_enhance(v, c)

        # Fusion (concatenate features then linear map back to D)
        f = (a + v) / 2.0  # simple fusion by averaging
        f = self.enc_fusion(f)

        # Per‑frame logits
        logit_a = self.head_a(a).squeeze(-1)  # (B,T)
        logit_v = self.head_v(v).squeeze(-1)
        logit_f = self.head_f(f).squeeze(-1)
        return logit_a, logit_v, logit_f

    # -------- Lightning training --------
    def training_step(self, batch, batch_idx):
        # Handle both old tuple format and new dict format
        if isinstance(batch, dict):
            # New compatible dataset format
            audio = batch['features']['audio']
            visual = batch['features']['visual'] 
            caption = batch['features']['caption']
            labels = batch['labels']
            
            # Apply sequence mask to ignore padded positions
            seq_mask = batch['sequence_masks']
            
            logit_a, logit_v, logit_f = self(audio, visual, caption)
            
            # Apply sequence mask to losses - only compute loss on valid positions
            valid_positions = seq_mask.bool()
            
            # Flatten and select valid positions
            logit_a_valid = logit_a[valid_positions]
            logit_v_valid = logit_v[valid_positions] 
            logit_f_valid = logit_f[valid_positions]
            labels_valid = labels[valid_positions]
            
            # focal losses on valid positions only
            loss_a = self.focal_loss(logit_a_valid, labels_valid)
            loss_v = self.focal_loss(logit_v_valid, labels_valid)
            loss_uni = loss_a + loss_v
            loss_mul = self.focal_loss(logit_f_valid, labels_valid)
            
            # alignment losses (KL divergence) on valid positions
            prob_a = torch.sigmoid(logit_a_valid).detach()
            prob_v = torch.sigmoid(logit_v_valid).detach()
            prob_f = torch.sigmoid(logit_f_valid)
            loss_kl = kl_div_bernoulli(prob_v, prob_f) + kl_div_bernoulli(prob_a, prob_f)
            
        else:
            # Old tuple format: audio, visual, caption, labels
            audio, visual, caption, labels = batch
            logit_a, logit_v, logit_f = self(audio, visual, caption)

            # focal losses
            loss_a = self.focal_loss(logit_a, labels)
            loss_v = self.focal_loss(logit_v, labels)
            loss_uni = loss_a + loss_v  # L_uni_focal (sum over two uni‑modal)
            loss_mul = self.focal_loss(logit_f, labels)  # L_mul_focal

            # alignment losses (KL divergence)
            prob_a = torch.sigmoid(logit_a).detach()
            prob_v = torch.sigmoid(logit_v).detach()
            prob_f = torch.sigmoid(logit_f)
            loss_kl = kl_div_bernoulli(prob_v, prob_f) + kl_div_bernoulli(prob_a, prob_f)

        # total loss
        total = LAMBDA1 * loss_uni + LAMBDA2 * loss_mul + LAMBDA3 * loss_kl

        self.log_dict({'loss_total': total,
                       'loss_uni': loss_uni,
                       'loss_mul': loss_mul,
                       'loss_kl': loss_kl}, prog_bar=True)
        return total

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# ==================== Train & plot ====================
# Get a sample batch to determine dimensions
sample_batch = next(iter(dataloader))
print(f"Sample batch keys: {sample_batch.keys()}")
print(f"Features keys: {sample_batch['features'].keys()}")

# Extract dimensions from the batch
dim_audio = sample_batch['features']['audio'].shape[-1]
dim_visual = sample_batch['features']['visual'].shape[-1] 
dim_caption = sample_batch['features']['caption'].shape[-1]

print(f"Dimensions - audio: {dim_audio}, visual: {dim_visual}, caption: {dim_caption}")

model = RepurposeModel(dim_audio=dim_audio,
                       dim_visual=dim_visual,
                       dim_caption=dim_caption)

class EpochLossCallback(pl.Callback):
    def __init__(self):
        self.losses = []
    def on_train_epoch_end(self, trainer, pl_module):
        self.losses.append(trainer.callback_metrics['loss_total'].item())

loss_cb = EpochLossCallback()

trainer = Trainer(max_epochs=EPOCHS, callbacks=[loss_cb],
                  log_every_n_steps=1, accelerator='auto')
trainer.fit(model, dataloader)

plt.plot(loss_cb.losses)
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
"""
Repurpose baseline (+memory‑friendly tweaks).
 - Added MemoryClearCallback: `gc.collect()` every epoch.
 - Disabled Lightning checkpointing / logging (eat RAM even on CPU).
 - DataLoader uses `num_workers=0` (workers fork copies of dataset that linger).
If RAM still creeps, uncomment the call to `torch.manual_gc()` (PyTorch ≥2.4).
"""

import gc
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from typing import Tuple

# ------------------- hyper‑params (unchanged) -------------------
D_MODEL = 128; N_HEAD = 16; N_LAYERS = 2; BATCH = 1
GAMMA, ALPHA = 2.0, 0.25
LAMBDA1, LAMBDA2, LAMBDA3 = 0.1, 0.3, 0.1
EPOCHS = 10

# ------------------- data -------------------
audio_list   = np.load('audio_list.npy',   allow_pickle=True)
caption_list = np.load('caption_list.npy', allow_pickle=True)
video_list   = np.load('video_list.npy',   allow_pickle=True)
labels_list  = np.load('labels_list.npy',  allow_pickle=True)

class SequenceDataset(Dataset):
    def __init__(self, a, c, v, lbl):
        self.items=[(torch.tensor(a_,dtype=torch.float32),
                     torch.tensor(v_,dtype=torch.float32),
                     torch.tensor(c_,dtype=torch.float32),
                     torch.tensor(l_,dtype=torch.float32))
                    for a_,c_,v_,l_ in zip(a,c,v,lbl)]
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

dataloader = DataLoader(SequenceDataset(audio_list,caption_list,video_list,labels_list),
                        batch_size=BATCH, shuffle=True, num_workers=0, persistent_workers=False)

# ------------------- focal & KL helpers -------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=ALPHA, gamma=GAMMA):
        super().__init__(); self.a, self.g = alpha, gamma
    def forward(self, logits, tgt):
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, tgt, reduction='none')
        p_t = p*tgt + (1-p)*(1-tgt)
        loss = (self.a*tgt + (1-self.a)*(1-tgt)) * (1-p_t)**self.g * ce
        return loss.mean()

def kl_div_bern(p,q,eps=1e-6):
    p=q.clamp(eps,1-eps),
    p,q=p[0],q.clamp(eps,1-eps)
    return (p*torch.log(p/q)+(1-p)*torch.log((1-p)/(1-q))).mean()

# ------------------- model (same arch) -------------------
class Repurpose(pl.LightningModule):
    def __init__(self, da,dv,dc):
        super().__init__(); D=D_MODEL
        self.proj_a, self.proj_v, self.proj_c = nn.Linear(da,D), nn.Linear(dv,D), nn.Linear(dc,D)
        def enc():
            return nn.TransformerEncoder(nn.TransformerEncoderLayer(D,N_HEAD,D*4,batch_first=True),N_LAYERS)
        self.enc_a,self.enc_v,self.enc_c=enc(),enc(),enc()
        self.enc_fusion=nn.TransformerEncoder(nn.TransformerEncoderLayer(D,N_HEAD,D*4,batch_first=True),1)
        def head(): return nn.Sequential(nn.Linear(D,D//2),nn.ReLU(),nn.Linear(D//2,1))
        self.head_a,self.head_v,self.head_f=head(),head(),head()
        self.focal=FocalLoss()
    def forward(self,a,v,c):
        a,v,c=self.proj_a(a),self.proj_v(v),self.proj_c(c)
        a,v,c=self.enc_a(a),self.enc_v(v),self.enc_c(c)
        a,v=a+c,v+c; f=self.enc_fusion((a+v)/2)
        return self.head_a(a).squeeze(-1),self.head_v(v).squeeze(-1),self.head_f(f).squeeze(-1)
    def training_step(self,b,_):
        a,v,c,l=b
        la,lv,lf=self(*b[:3])
        lu=self.focal(la,l)+self.focal(lv,l)
        lm=self.focal(lf,l)
        kl=kl_div_bern(torch.sigmoid(la.detach()),torch.sigmoid(lf))+kl_div_bern(torch.sigmoid(lv.detach()),torch.sigmoid(lf))
        loss=LAMBDA1*lu+LAMBDA2*lm+LAMBDA3*kl
        self.log('loss',loss,prog_bar=True)
        return loss
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(),lr=1e-3)

# ------------------- callbacks -------------------
class ClearMem(pl.Callback):
    def on_train_epoch_end(self,tr,plm):
        gc.collect()
        torch.manual_gc()  # uncomment if you have PyTorch≥2.4

class LossPlot(pl.Callback):
    def __init__(self): self.hist=[]
    def on_train_epoch_end(self,tr,plm): self.hist.append(tr.callback_metrics['loss'].item())
    def on_train_end(self,tr,plm):
        plt.plot(self.hist); plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()

# ------------------- train -------------------
samp=next(iter(dataloader))
model=Repurpose(samp[0].shape[-1],samp[1].shape[-1],samp[2].shape[-1])
trainer=Trainer(max_epochs=EPOCHS, callbacks=[ClearMem(),LossPlot()], accelerator='cpu', enable_checkpointing=False, logger=False)
trainer.fit(model,dataloader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
   | Name       | Type               | Params | Mode 
-----------------------------------------------------------
0  | proj_a     | Linear             | 262 K  | train
1  | proj_v     | Linear             | 65.7 K | train
2  | proj_c     | Linear             | 49.3 K | train
3  | enc_a      | TransformerEncoder | 396 K  | train
4  | enc_v      | TransformerEncoder | 396 K  | train
5  | enc_c      | TransformerEncoder | 396 K  | train
6  | enc_fusion | TransformerEncoder | 198 K  | train
7  | head_a     | Sequential         | 8.3 K  | train
8  | head_v     | Sequential         | 8.3 K  | train
9  | head_f     | Sequential         | 8.3 K  | train
10 | focal      | FocalLoss          | 0      | train
----

Training: |          | 0/? [00:00<?, ?it/s]

In [ ]:
"""
Visualization module for model predictions and debug outputs.
Based on debug_visualizer.py - includes visualization of predictions vs ground truth.
"""

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from typing import List, Tuple, Optional

def visualize_model_predictions(
    model: pl.LightningModule,
    dataloader: DataLoader,
    num_samples: int = 5,
    save_dir: Optional[str] = None,
    device: str = 'cpu'
) -> List[str]:
    """
    Visualize model predictions vs ground truth for debugging.
    
    Args:
        model: Trained PyTorch Lightning model
        dataloader: DataLoader with test samples
        num_samples: Number of samples to visualize
        save_dir: Directory to save visualizations
        device: Device to run inference on
    
    Returns:
        List of saved figure paths
    """
    model.eval()
    model = model.to(device)
    
    saved_paths = []
    
    with torch.no_grad():
        for idx, batch in enumerate(dataloader):
            if idx >= num_samples:
                break
                
            # Get data
            if len(batch) == 4:  # audio, visual, caption, labels
                audio, visual, caption, labels = batch
                audio = audio.to(device)
                visual = visual.to(device)
                caption = caption.to(device)
                labels = labels.to(device)
                
                # Get predictions
                if hasattr(model, 'forward'):
                    # For RepurposeModel
                    logit_a, logit_v, logit_f = model(audio, visual, caption)
                    # Use fused predictions for visualization
                    logits = logit_f
                else:
                    # For simple MLP model
                    x = torch.cat([audio, visual, caption], dim=-1)
                    logits = model(x)
            else:
                # Simple x, y batch
                x, y = batch
                x = x.to(device)
                labels = y.to(device)
                logits = model(x)
            
            # Convert to numpy
            pred_probs = torch.sigmoid(logits).cpu().numpy()
            labels_np = labels.cpu().numpy()
            
            # Handle batch dimension
            if pred_probs.ndim == 2:
                pred_probs = pred_probs[0]  # Take first sample in batch
                labels_np = labels_np[0]
            
            # Create visualization
            fig, axes = plt.subplots(3, 1, figsize=(15, 10))
            
            seq_len = len(pred_probs)
            time_points = np.arange(seq_len)
            
            # Plot 1: Classification scores
            ax1 = axes[0]
            ax1.plot(time_points, pred_probs, 'b-', label='Predicted Prob', alpha=0.7)
            
            # Mark positive ground truth points
            positive_idx = labels_np > 0.5
            if np.any(positive_idx):
                ax1.scatter(time_points[positive_idx], 
                           np.ones(np.sum(positive_idx)), 
                           color='red', s=50, label='GT Positive', zorder=5)
            
            ax1.set_ylabel('Classification Score')
            ax1.set_title(f'Sample {idx} - Classification Predictions')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            ax1.set_ylim(-0.1, 1.1)
            
            # Plot 2: Prediction confidence over time
            ax2 = axes[1]
            confidence = np.abs(pred_probs - 0.5) * 2  # Distance from decision boundary
            ax2.plot(time_points, confidence, 'g-', label='Confidence', alpha=0.7)
            ax2.set_ylabel('Confidence')
            ax2.set_title('Prediction Confidence (distance from 0.5)')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim(0, 1)
            
            # Plot 3: Segments visualization
            ax3 = axes[2]
            
            # Draw predicted segments (consecutive high confidence regions)
            in_segment = False
            segment_start = 0
            threshold = 0.5
            
            for t in range(seq_len):
                if pred_probs[t] > threshold and not in_segment:
                    in_segment = True
                    segment_start = t
                elif pred_probs[t] <= threshold and in_segment:
                    in_segment = False
                    ax3.add_patch(patches.Rectangle(
                        (segment_start, 0.6), t - segment_start, 0.3,
                        facecolor='blue', alpha=0.5, label='Predicted' if segment_start == 0 else ""
                    ))
            
            # Handle case where segment extends to end
            if in_segment:
                ax3.add_patch(patches.Rectangle(
                    (segment_start, 0.6), seq_len - segment_start, 0.3,
                    facecolor='blue', alpha=0.5
                ))
            
            # Draw GT segments
            in_gt_segment = False
            gt_segment_start = 0
            
            for t in range(seq_len):
                if labels_np[t] > 0.5 and not in_gt_segment:
                    in_gt_segment = True
                    gt_segment_start = t
                elif labels_np[t] <= 0.5 and in_gt_segment:
                    in_gt_segment = False
                    ax3.add_patch(patches.Rectangle(
                        (gt_segment_start, 0.1), t - gt_segment_start, 0.3,
                        facecolor='red', alpha=0.5, label='Ground Truth' if gt_segment_start == 0 else ""
                    ))
            
            # Handle case where GT segment extends to end
            if in_gt_segment:
                ax3.add_patch(patches.Rectangle(
                    (gt_segment_start, 0.1), seq_len - gt_segment_start, 0.3,
                    facecolor='red', alpha=0.5
                ))
            
            ax3.set_xlim(0, seq_len)
            ax3.set_ylim(0, 1)
            ax3.set_xlabel('Time Steps')
            ax3.set_title('Segment Visualization (Blue: Predicted, Red: Ground Truth)')
            ax3.grid(True, alpha=0.3, axis='x')
            
            # Add simple legend
            blue_patch = patches.Patch(color='blue', alpha=0.5, label='Predicted')
            red_patch = patches.Patch(color='red', alpha=0.5, label='Ground Truth')
            ax3.legend(handles=[blue_patch, red_patch])
            
            plt.tight_layout()
            
            # Save or show
            if save_dir:
                import os
                os.makedirs(save_dir, exist_ok=True)
                path = os.path.join(save_dir, f'visualization_sample_{idx}.png')
                plt.savefig(path, dpi=150, bbox_inches='tight')
                saved_paths.append(path)
                print(f"Saved visualization to {path}")
            else:
                plt.show()
            
            plt.close()
    
    return saved_paths


def analyze_predictions(
    model: pl.LightningModule,
    dataloader: DataLoader,
    device: str = 'cpu'
) -> dict:
    """
    Analyze model predictions and return statistics.
    """
    model.eval()
    model = model.to(device)
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            if len(batch) == 4:  # audio, visual, caption, labels
                audio, visual, caption, labels = batch
                audio = audio.to(device)
                visual = visual.to(device)
                caption = caption.to(device)
                
                # Get predictions
                if hasattr(model, 'forward'):
                    _, _, logit_f = model(audio, visual, caption)
                    logits = logit_f
                else:
                    x = torch.cat([audio, visual, caption], dim=-1)
                    logits = model(x)
            else:
                x, y = batch
                x = x.to(device)
                labels = y
                logits = model(x)
            
            pred_probs = torch.sigmoid(logits)
            all_preds.extend(pred_probs.cpu().numpy().flatten())
            all_labels.extend(labels.cpu().numpy().flatten())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate statistics
    stats = {
        'total_predictions': len(all_preds),
        'positive_labels': np.sum(all_labels > 0.5),
        'negative_labels': np.sum(all_labels <= 0.5),
        'positive_ratio': np.mean(all_labels > 0.5),
        'pred_stats': {
            'min': np.min(all_preds),
            'max': np.max(all_preds),
            'mean': np.mean(all_preds),
            'std': np.std(all_preds),
            'positive_predictions': np.sum(all_preds > 0.5),
            'positive_pred_ratio': np.mean(all_preds > 0.5)
        }
    }
    
    # Calculate simple accuracy metrics
    binary_preds = (all_preds > 0.5).astype(float)
    binary_labels = (all_labels > 0.5).astype(float)
    
    correct = np.sum(binary_preds == binary_labels)
    accuracy = correct / len(all_labels)
    
    # True positives, false positives, etc.
    tp = np.sum((binary_preds == 1) & (binary_labels == 1))
    fp = np.sum((binary_preds == 1) & (binary_labels == 0))
    tn = np.sum((binary_preds == 0) & (binary_labels == 0))
    fn = np.sum((binary_preds == 0) & (binary_labels == 1))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    stats['metrics'] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'true_positives': int(tp),
        'false_positives': int(fp),
        'true_negatives': int(tn),
        'false_negatives': int(fn)
    }
    
    return stats


# Example usage:
print("Visualization functions loaded!")
print("\nTo visualize predictions:")
print("visualize_model_predictions(model, dataloader, num_samples=3)")
print("\nTo analyze predictions:")
print("stats = analyze_predictions(model, dataloader)")
print("print(stats)")

In [ ]:
# Example: Visualize model predictions
# Uncomment and run after training a model

# For the RepurposeModel (transformer-based)
# visualize_model_predictions(model, dataloader, num_samples=3, save_dir='visualizations')

# Analyze prediction statistics
# stats = analyze_predictions(model, dataloader)
# print("\n=== Prediction Analysis ===")
# print(f"Total predictions: {stats['total_predictions']}")
# print(f"Positive label ratio: {stats['positive_ratio']:.2%}")
# print(f"\nPrediction statistics:")
# for key, value in stats['pred_stats'].items():
#     print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
# print(f"\nPerformance metrics:")
# for key, value in stats['metrics'].items():
#     print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")